# Linear Operators

Linear operators are the computational backbone of Furax, providing composable building blocks for solving inverse problems in CMB analysis. Built on top of JAX, Furax operators support mathematical composition, automatic differentiation, and efficient GPU computation.

The examples of this page run in 64-bit precision and share the following imports:

In [ ]:
import jax

jax.config.update('jax_enable_x64', True)

In [ ]:
import jax.numpy as jnp
import jax.random as jr
import lineax as lx

from furax import (
    BlockColumnOperator,
    BlockDiagonalOperator,
    BlockRowOperator,
    BroadcastDiagonalOperator,
    DenseBlockDiagonalOperator,
    DiagonalOperator,
    IndexOperator,
    MoveAxisOperator,
    ReshapeOperator,
    SymmetricBandToeplitzOperator,
    TreeOperator,
)
from furax.obs.landscapes import HealpixLandscape
from furax.obs.stokes import StokesQU

## Core Concepts

### Abstract Linear Operator

All operators in Furax inherit from `AbstractLinearOperator`, a frozen dataclass abstract base class:

- **Composition**: Operators can be composed using the `@` operator (matrix multiplication)
- **Addition**: Operators can be added using the `+` operator
- **Scalar Operations**: Support for scalar multiplication and division
- **Matrix Representation**: Convert to explicit matrices for debugging with `as_matrix()`
- **Properties**: Static mathematical properties set via decorators (`@symmetric`, `@diagonal`, etc.)

In [ ]:
# Create operators
op1 = DiagonalOperator(jnp.array([1.0, 2.0, 3.0]))
op2 = DiagonalOperator(jnp.array([2.0, 1.0, 1.0]))

# Composition (matrix multiplication)
composed = op1 @ op2

# Addition
summed = op1 + op2

# Scalar operations
scaled = 2.5 * op1
divided = op1 / 3.0

# Operators have tags, which are determined statically (not inspecting values): a diagonal
# operator is not tagged positive semi-definite, even though its values are positive
print(f'Is symmetric: {op1.is_symmetric}')
print(f'Is positive semi-definite: {op1.is_positive_semidefinite}')

## Operator Types

### Diagonal Operators

Perfect for pixel-based weighting, noise covariance, and preconditioning.

**DiagonalOperator**

In [ ]:
# Create a diagonal operator for weighting
weights = jnp.array([1.0, 0.5, 2.0, 1.5])
weight_op = DiagonalOperator(weights)

# Apply to data
data = jnp.array([1.0, 2.0, 3.0, 4.0])
weight_op(data)

**BroadcastDiagonalOperator**

For operations that need broadcasting across multiple dimensions:

In [ ]:
# Diagonal values to broadcast
diag_values = jnp.array([1.0, 2.0, 3.0])

# Create operator that broadcasts to (3, 4) arrays
broadcast_op = BroadcastDiagonalOperator(
    diagonal=diag_values,
    axis_destination=0,  # Broadcast along axes except first one
    in_structure=jax.ShapeDtypeStruct((3, 4), jnp.float64),
)

# Apply to multi-dimensional data: each row is scaled by the corresponding diagonal value
data = jnp.ones((3, 4))
broadcast_op(data)

### Block Operators

Essential for multi-component analysis and structured linear systems. The blocks can be assembled using
any Python container, as long as it is a PyTree. They can also be an
[`Ensemble`][furax.core.Ensemble] of operators, which batches and distributes the blocks: see
[Distributing work](distributing-work.md).

**BlockDiagonalOperator**

In [ ]:
# Create individual block operators
block1 = DiagonalOperator(jnp.array([1.0, 2.0]))
block2 = DiagonalOperator(jnp.array([3.0, 4.0, 5.0]))
block3 = DiagonalOperator(jnp.array([6.0]))

# Create block diagonal operator using a tuple of operators
block_op = BlockDiagonalOperator((block1, block2, block3))
block_op.as_matrix()

In [ ]:
# Apply to a tuple of data
data = (jnp.array([1.0, 1.0]), jnp.array([1.0, 1.0, 1.0]), jnp.array([1.0]))
block_op(data)

**BlockRowOperator**

For horizontal concatenation `[A B C]`:

In [ ]:
op1 = DiagonalOperator(jnp.array([1.0, 2.0]))
op2 = DenseBlockDiagonalOperator(
    jnp.array([[3.0, 0.0, 4.0], [0.0, 5.0, 0.0]]),
    in_structure=jax.ShapeDtypeStruct((3,), jnp.float64),
)

# Create row block: [op1, op2]
row_op = BlockRowOperator([op1, op2])
row_op.as_matrix()

In [ ]:
# Input has combined size
data = [jnp.array([1.0, 1.0]), jnp.array([1.0, 1.0, 1.0])]
row_op(data)

**BlockColumnOperator**

For vertical stacking:

In [ ]:
op1 = DiagonalOperator(jnp.array([1.0, 2.0]))
op2 = DiagonalOperator(jnp.array([3.0, 4.0]))

# Create column block
col_op = BlockColumnOperator({'x': op1, 'y': op2})
col_op.as_matrix()

In [ ]:
data = jnp.array([1.0, 1.0])
col_op(data)

### Dense Operators on Stokes Parameters

`DenseBlockDiagonalOperator` applies a stack of small dense matrices with `jnp.einsum`. It is the
natural way to mix Stokes parameters, for instance with a rotation or a Mueller matrix, possibly
different in each pixel or frequency.

A `Stokes` container stores its components in a single array, stacked on the leading axis: a
`StokesQU` map of 3 pixels holds an array of shape `(2, 3)`, with Q in row 0 and U in row 1.
The operator applies the einsum to this array, and wraps the result in the same Stokes class. In the
subscripts, the first axis of the input and of the output is therefore the Stokes axis.

**Same matrix in every pixel**

With the default subscripts `'ij...,j...->i...'`, the matrix contracts the Stokes axis and is
broadcast over the remaining axes:

In [ ]:
x = StokesQU.from_stokes(q=jnp.array([1.0, 2.0, 3.0]), u=jnp.zeros(3))

# Rotation of the polarization angle by 45 degrees
rotation = jnp.array([[0.0, -1.0], [1.0, 0.0]])
op = DenseBlockDiagonalOperator(rotation, in_structure=x.structure)
y = op(x)
y.q, y.u

**One matrix per pixel**

A pixel axis that appears in the matrix, the input and the output, without being summed over, makes
the operator block diagonal in pixels. Here `s` is the output Stokes axis, `t` the input Stokes axis
and `p` the pixel axis:

In [ ]:
psi = jnp.array([0.0, jnp.pi / 4, jnp.pi / 2])
c, s = jnp.cos(2 * psi), jnp.sin(2 * psi)
rotations = jnp.array([[c, -s], [s, c]])  # shape (2, 2, 3)
op = DenseBlockDiagonalOperator(rotations, in_structure=x.structure, subscripts='stp,tp->sp')
y = op(x)
y.q, y.u

In [ ]:
# The transpose rotates back
x_back = op.T(y)
x_back.q, x_back.u

**Several contracted axes**

The matrix can also mix pixels. In the example below, maps of `npix_in` pixels at each frequency are
mapped to `npix_out` pixels, with a matrix that couples Q and U. The einsum contracts both the input
Stokes axis `t` and the input pixel axis `p`. The axes of the matrix can come in any order:

In [ ]:
nfreq, npix_in, npix_out = 2, 3, 4
x = StokesQU.ones((nfreq, npix_in))
matrix = jnp.ones((nfreq, 2, 2, npix_out, npix_in))  # (f, s, t, q, p)
op = DenseBlockDiagonalOperator(matrix, in_structure=x.structure, subscripts='fstqp,tfp->sfq')
op.out_structure

The transpose reuses the same matrix with the input and output subscripts swapped, here
`'fstqp,sfq->tfp'`. It maps `StokesQU` maps of shape `(nfreq, npix_out)` back to
`(nfreq, npix_in)`.

!!! warning
    The output always has the same Stokes class as the input, so the Stokes axis must keep its size.
    A `(2, 3)` matrix meant to project IQU onto QU would return a `StokesIQU` holding only two
    components, without raising an error. To drop components, use a square matrix with zero rows
    instead.

### Toeplitz Operators

Efficient for convolution-like operations and correlated noise modeling.

In [ ]:
# Define the bands for a symmetric Toeplitz matrix
bands = jnp.array([[2.0, 1.0, 0.5], [1, 0.8, 0.1]])

# Create symmetric band Toeplitz operator
toeplitz_op = SymmetricBandToeplitzOperator(
    bands, in_structure=jax.ShapeDtypeStruct((2, 6), jnp.float64)
)
toeplitz_op.as_matrix()

In [ ]:
# Apply to data
data = jnp.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0])
toeplitz_op(data)

By default, the operator convolves by FFT (overlap-save method), hence the round-off errors at the floating-point precision level; `method='direct'` computes the band products directly.

### Index and Reshape Operators

For data manipulation and restructuring.

**IndexOperator**

In [ ]:
# Select specific indices
indices = jnp.array([0, 2, 4])
index_op = IndexOperator(indices, in_structure=jax.ShapeDtypeStruct((5,), jnp.float64))

data = jnp.array([10.0, 20.0, 30.0, 40.0, 50.0])
index_op(data)

**ReshapeOperator**

In [ ]:
# Reshape from (6,) to (2, 3)
reshape_op = ReshapeOperator(
    shape=(2, 3),
    in_structure={'x': jax.ShapeDtypeStruct((6,), jnp.float64)},
)

data = {'x': jnp.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])}
reshape_op(data)

**MoveAxisOperator**

In [ ]:
# Move axis from position 0 to position 1
moveaxis_op = MoveAxisOperator(
    source=0, destination=1, in_structure=[jax.ShapeDtypeStruct((6, 2), jnp.float64)]
)

data = [jnp.arange(12.0).reshape((6, 2))]
moveaxis_op(data)

### Tree Operators

`TreeOperator` represents a generalized matrix as a pytree of pytrees: the outer tree gives the rows, that is the output structure, and the inner tree the columns, matching the input structure. Each entry multiplies the input leaf of its column, with broadcasting, and the products of a row are summed. Here, a rotation of the polarization angle acts on maps stored as a dictionary of Q and U arrays:

In [ ]:
psi = jnp.pi / 8
c, s = jnp.cos(2 * psi), jnp.sin(2 * psi)

# Rows: the output Q and U; columns: the input Q and U
rotation_op = TreeOperator(
    {'q': {'q': c, 'u': -s}, 'u': {'q': s, 'u': c}},
    in_structure={
        'q': jax.ShapeDtypeStruct((3,), jnp.float64),
        'u': jax.ShapeDtypeStruct((3,), jnp.float64),
    },
)

data = {'q': jnp.ones(3), 'u': jnp.zeros(3)}
rotation_op(data)

## Advanced Operator Composition

### Complex Analysis Pipelines

Operators can be composed to create sophisticated analysis pipelines:

In [ ]:
# Create a landscape for QU polarization
landscape = HealpixLandscape(nside=8, stokes='QU')
n_pix = landscape.shape[0]

# 1. Noise weighting (inverse variance), the same for Q and U
noise_var = jnp.ones(n_pix)
noise_weighting = DiagonalOperator(1.0 / noise_var, in_structure=landscape.structure)

# 2. Pixel selection (mask bad pixels) on the last axis, since the Stokes components lead
good_pixels = jnp.arange(n_pix)[::2]  # Select every other pixel
pixel_selection = IndexOperator((..., good_pixels), in_structure=landscape.structure)

# 3. Component-wise processing, with weights stacked like the Stokes maps
component_weights = jnp.stack([jnp.ones(n_pix), 2.0 * jnp.ones(n_pix)])
component_processor = DiagonalOperator(component_weights, in_structure=landscape.structure)

# Compose the full pipeline
analysis_pipeline = pixel_selection @ component_processor @ noise_weighting

# Apply to data
data = landscape.normal(jr.key(0))
processed_data = analysis_pipeline(data)
processed_data.structure

### Iterative Solvers

Furax operators work seamlessly with jax.scipy and lineax solvers:

In [ ]:
# Create a positive definite operator for solving Ax = b
band = jnp.array([1.0, 0.5, 0.25, 0.125])
A = SymmetricBandToeplitzOperator(band, in_structure=jax.ShapeDtypeStruct((6,), jnp.float64))

# Right-hand side
b = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])

# Solve with conjugate gradient
solver = lx.CG(atol=1e-10, rtol=1e-10, max_steps=100)
solution = A.I(solver=solver)(b)
print(f'CG solution: {solution}')

# Solve with GMRES
solution = jax.scipy.sparse.linalg.gmres(A, b)[0]
print(f'GMRES solution: {solution}')

expected_solution = jnp.linalg.inv(A.as_matrix()) @ b
print(f'Expected solution: {expected_solution}')

### Matrix-Free Operations

Operators support matrix-free computations for memory efficiency:

In [ ]:
def large_scale_analysis(operator, data):
    """Perform analysis without forming explicit matrices."""
    # Matrix-vector product (never forms the full matrix)
    result = operator(data)

    # Operator norms and properties
    print('Operator properties:')
    print(f'  Symmetric: {operator.is_symmetric}')
    print(f'  Positive semidefinite: {operator.is_positive_semidefinite}')

    return result


# Even for very large operators, memory usage stays manageable
large_diagonal = DiagonalOperator(jnp.ones(1_000_000))
large_data = jnp.ones(1_000_000)

result = large_scale_analysis(large_diagonal, large_data)

## Operator Properties

### Mathematical Properties

Furax statically stores algebraic properties of operators (such as squareness, symmetry, orthogonality, ...). Note that these properties are not inferred from the operator's
data since it would not work after jitting the operation. These properties are not yet propagated during composition
(but note that most properties such as symmetry, positive semi-definiteness are not preserved in general)

In [ ]:
# Diagonal operators are tagged as symmetric but not as PSD (tags are static, not value-dependent)
positive_diag = DiagonalOperator(jnp.array([1.0, 2.0, 3.0]))
print(f'Square: {positive_diag.is_square}')
print(f'Symmetric: {positive_diag.is_symmetric}')
print(f'Positive semidefinite: {positive_diag.is_positive_semidefinite}')

### Custom Operators

You can create custom operators by inheriting from `AbstractLinearOperator`:

In [ ]:
from dataclasses import field

from jax import Array
from jaxtyping import Inexact, PyTree

from furax import AbstractLinearOperator, symmetric


@symmetric
class CustomScalingOperator(AbstractLinearOperator):
    """Custom operator that scales a PyTree by a static factor."""

    scale_factor: float = field(metadata={'static': True})

    def mv(self, x: PyTree[Inexact[Array, ' _a']]) -> PyTree[Inexact[Array, ' _b']]:
        return jax.tree.map(lambda leaf: self.scale_factor * leaf, x)


# Use the custom operator
custom_op = CustomScalingOperator(
    scale_factor=2.5, in_structure={'input': jax.ShapeDtypeStruct((3,), jnp.float64)}
)
data = {'input': jnp.array([1.0, 2.0, 3.0])}
custom_op(data)

In [ ]:
# The operator is symmetric
print(custom_op.is_symmetric)
assert custom_op.T is custom_op

# Square operators can be inverted (default: CG)
custom_op.I(data)

## Performance Considerations

### JAX Transformations

Operators work efficiently with JAX transformations. Since operators are PyTrees, they can be jitted, vmapped etc.

In [ ]:
# JIT compilation
@jax.jit
def fast_operator_apply(op, data):
    return op(data)


op = DiagonalOperator(jnp.array([1.0, 2.0, 3.0, 4.0]))
data = jnp.array([1.0, 1.0, 1.0, 1.0])

# First call compiles, subsequent calls are fast
fast_operator_apply(op, data)

In [ ]:
# Vectorization
@jax.vmap
def batch_apply(data_batch):
    return op(data_batch)


# Apply operator to batch of data
data_batch = jnp.arange(40.0).reshape(10, 4)  # 10 samples of size 4
batch_apply(data_batch)

### Memory Efficiency

For large-scale problems:

1. **Use appropriate operator types**: Diagonal operators are more memory-efficient than dense operators
2. **Avoid explicit matrix formation**: Use `operator(data)` instead of `operator.as_matrix() @ data`
3. **Consider block structure**: Block operators can reduce memory usage for structured problems
4. **Use appropriate precision**: Float32 vs Float64 trade-offs
5. **Distribute repeated work**: when many operands share one structure — the observations of a
   season, the blocks of a composite — see [Distributing work](distributing-work.md)


## Best Practices

1. **Compose operators logically**: Build complex operations from simple, well-understood components

2. **Leverage mathematical properties**: Use symmetric, positive definite operators when possible for better solver performance

3. **Test with small examples**: Verify operator behavior with `as_matrix()` on small problems

4. **Profile memory usage**: For large problems, monitor memory consumption

5. **Use appropriate solvers**: Match solver choice to operator properties (e.g., CG for symmetric positive definite systems)

6. **Batch operations**: Use `vmap` to process multiple datasets efficiently